In [3]:
import os
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter


# =========================
# Excel保存路径
# =========================

desktop = os.path.join(
    os.path.expanduser("~"),
    "Desktop"
)

output_file = os.path.join(
    desktop,
    "小红书笔记数据.xlsx"
)


# 配置 Chrome 浏览器选项
options = Options()
options.add_argument(r"--user-data-dir=C:\Users\qieziclub1660ti\AppData\Local\Google\Chrome\SeleniumData")  # 替换为实际用户数据目录
options.add_argument("--start-maximized")  # 最大化窗口

# 创建 Chrome WebDriver
webdriver_path = r"C:\WebDriver\chromedriver.exe"

driver = webdriver.Chrome(
    service=Service(webdriver_path),
    options=options
)



# =========================
# 打开小红书创作中心
# =========================

url = "https://creator.xiaohongshu.com/new/note-manager"

try:

    driver.get(url)

    print("成功进入小红书创作中心")


except Exception as e:

    print(f"打开页面失败：{e}")

    driver.quit()

    exit()


# =========================
# 等待登录
# =========================

wait = WebDriverWait(driver, 300)

try:

    print("请扫码登录小红书创作中心...")

    wait.until(
        EC.presence_of_element_located(
            (
                By.CSS_SELECTOR,
                "div.note-card"
            )
        )
    )

    print("登录成功，进入笔记管理页面")


except Exception as e:

    print(f"登录失败或等待超时：{e}")

    driver.quit()

    exit()


# =========================
# 数字转换函数
# =========================

def convert_number(text):

    if not text:
        return 0

    text = text.strip()

    # 处理短横线等无数据情况
    if text in ["-", "--"]:
        return 0

    # 处理“万”
    if "万" in text:

        try:

            return int(
                float(
                    text.replace("万", "")
                ) * 10000
            )

        except ValueError:

            return 0

    # 处理逗号
    try:

        return int(
            text.replace(",", "")
        )

    except ValueError:

        return 0


# =========================
# 保存全部笔记
# =========================

all_note_data = []

# 用标题和日期共同去重
loaded_notes = set()

last_total = 0

no_change_count = 0


# =========================
# 自动滚动并提取全部笔记
# =========================

while True:

    # 查找当前已经加载的全部笔记卡片
    note_cards = driver.find_elements(
        By.CSS_SELECTOR,
        "div.note-card"
    )

    print(
        f"\n当前页面发现笔记：{len(note_cards)}"
    )


    for card in note_cards:

        try:

            # =========================
            # 提取标题
            # =========================

            title = card.find_element(
                By.CSS_SELECTOR,
                "span.note-card__title"
            ).text.strip()


            # =========================
            # 提取日期
            # =========================

            date = card.find_element(
                By.CSS_SELECTOR,
                "span.note-card__time"
            ).text.strip()


            if not title:

                continue


            # 标题和日期组合，防止同名笔记被误删
            unique_key = (
                title,
                date
            )


            if unique_key in loaded_notes:

                continue


            # =========================
            # 提取五项数据
            # =========================

            stat_elements = card.find_elements(
                By.CSS_SELECTOR,
                "div.note-card__stat span"
            )


            stat_values = [
                element.text.strip()
                for element in stat_elements
            ]


            # 默认全部为0
            view_count = 0
            comment_count = 0
            like_count = 0
            favorite_count = 0
            share_count = 0


            # 根据截图中图标顺序提取
            if len(stat_values) >= 1:

                view_count = convert_number(
                    stat_values[0]
                )


            if len(stat_values) >= 2:

                comment_count = convert_number(
                    stat_values[1]
                )


            if len(stat_values) >= 3:

                like_count = convert_number(
                    stat_values[2]
                )


            if len(stat_values) >= 4:

                favorite_count = convert_number(
                    stat_values[3]
                )


            if len(stat_values) >= 5:

                share_count = convert_number(
                    stat_values[4]
                )


            loaded_notes.add(
                unique_key
            )


            all_note_data.append(
                {
                    "笔记名称": title,
                    "发布日期": date,
                    "观看量": view_count,
                    "评论量": comment_count,
                    "点赞量": like_count,
                    "收藏量": favorite_count,
                    "分享量": share_count
                }
            )


            print(
                f"{title} | "
                f"日期：{date} | "
                f"观看：{view_count} | "
                f"评论：{comment_count} | "
                f"点赞：{like_count} | "
                f"收藏：{favorite_count} | "
                f"分享：{share_count}"
            )


        except Exception as e:

            print(
                f"单篇笔记提取失败：{e}"
            )


    # =========================
    # 滚动加载下一批
    # =========================

    scroll_result = driver.execute_script("""

    let elements = document.querySelectorAll("*");

    let target = null;

    let maxHeight = 0;


    for(let el of elements){

        let style = window.getComputedStyle(el);


        if(
            (style.overflowY === "auto" ||
            style.overflowY === "scroll")
            &&
            el.scrollHeight > el.clientHeight
        ){

            if(el.scrollHeight > maxHeight){

                maxHeight = el.scrollHeight;

                target = el;

            }

        }

    }


    if(target){

    let before = target.scrollTop;


    target.scrollTop = target.scrollTop + target.clientHeight;


    return {

        before:before,

        after:target.scrollTop,

        height:target.scrollHeight

    };

}


    return null;


    """)


    print(
        "滚动结果:",
        scroll_result
    )


    time.sleep(0.5)



    # 判断是否继续

    current_total = len(all_note_data)


    print(
        f"当前累计笔记数量:{current_total}"
    )


    if current_total == last_total:

        no_change_count += 1

    else:

        no_change_count = 0


    last_total = current_total



    if no_change_count >= 3:

        print(
            "连续3次没有新数据，结束"
        )

        break




# =========================
# 关闭浏览器
# =========================

driver.quit()


# =========================
# 保存为DataFrame
# =========================

print(
    f"\n最终提取笔记数量：{len(all_note_data)}"
)


if not all_note_data:

    print("没有提取到任何笔记数据")

    exit()


df = pd.DataFrame(
    all_note_data
)


# 将数字列强制转换为整数
number_columns = [
    "观看量",
    "评论量",
    "点赞量",
    "收藏量",
    "分享量"
]

for column in number_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    ).fillna(0).astype(int)


# =========================
# 保存Excel
# =========================

df.to_excel(
    output_file,
    index=False,
    sheet_name="笔记数据"
)


# =========================
# 调整Excel格式
# =========================

wb = load_workbook(
    output_file
)

ws = wb.active


# 设置列宽
ws.column_dimensions["A"].width = 45
ws.column_dimensions["B"].width = 22

for column_number in range(
    3,
    ws.max_column + 1
):

    column_letter = get_column_letter(
        column_number
    )

    ws.column_dimensions[
        column_letter
    ].width = 12


# 标题行居中
for cell in ws[1]:

    cell.alignment = Alignment(
        horizontal="center",
        vertical="center"
    )


# 日期和数字右对齐
for row in ws.iter_rows(
    min_row=2,
    min_col=2,
    max_col=ws.max_column,
    max_row=ws.max_row
):

    for cell in row:

        cell.alignment = Alignment(
            horizontal="right",
            vertical="center"
        )


wb.save(
    output_file
)


print(
    f"小红书笔记数据已保存到：{output_file}"
)


# =========================
# 自动打开Excel
# =========================

try:

    os.startfile(
        output_file
    )

except Exception as e:

    print(
        f"无法自动打开Excel：{e}"
    )

成功进入小红书创作中心
请扫码登录小红书创作中心...
登录成功，进入笔记管理页面

当前页面发现笔记：10
OCS需求爆了！MEMS微镜凭何是谷歌最优解？ | 日期：2026-07-13 18:33 | 观看：463 | 评论：0 | 点赞：9 | 收藏：16 | 分享：7
半导体气体产业链创新成果路演圆满落幕！ | 日期：2026-07-07 16:55 | 观看：244 | 评论：0 | 点赞：0 | 收藏：0 | 分享：0
硅光大爆发，与广立微携手抢占产业新机遇 | 日期：2026-07-01 17:12 | 观看：275 | 评论：0 | 点赞：7 | 收藏：7 | 分享：2
出席IPO敲钟仪式！幻实见证9年老友上市之路 | 日期：2026-06-24 18:25 | 观看：260 | 评论：0 | 点赞：3 | 收藏：1 | 分享：1
零失效+高交付 晶能领跑车规碳化硅主驱赛道 | 日期：2026-06-16 12:48 | 观看：277 | 评论：0 | 点赞：7 | 收藏：4 | 分享：1
揭秘！打卡国内首条光子芯片中试线！ | 日期：2026-06-11 10:47 | 观看：1 | 评论：0 | 点赞：0 | 收藏：0 | 分享：0
一家设计院的自我革命半导体 AI | 日期：2026-06-02 10:11 | 观看：3 | 评论：0 | 点赞：1 | 收藏：1 | 分享：0
跳出单点局限，广立微解锁AI+EDA新战力！ | 日期：2026-05-26 18:07 | 观看：967 | 评论：2 | 点赞：23 | 收藏：25 | 分享：21
不止学术交流！半导体圈隐藏的资源聚集地 | 日期：2026-05-20 09:15 | 观看：333 | 评论：0 | 点赞：10 | 收藏：8 | 分享：2
今年最硬核化合物半导体展会，幻实替你探了 | 日期：2026-04-25 12:17 | 观看：156 | 评论：0 | 点赞：5 | 收藏：1 | 分享：0
滚动结果: {'after': 327, 'before': 0, 'height': 1424}
当前累计笔记数量:10

当前页面发现笔记：30
半导体离不开这口气！幻实直击2026气博会 | 日期：2026-04-18 15:11 | 观看：224 | 评论：1 | 点赞：3 | 收藏